## Data Collection: U.S. Economy Videos and Comments from YouTube

### Project Overview

This notebook builds the initial dataset for a text and media analysis project focused on U.S. economy-related content on YouTube.

The goal is to identify topic-relevant videos from selected news channels, collect metadata and comment-level data, and structure the results into an analysis-ready dataset for downstream cleaning, exploratory analysis, and text mining.

### Keyword Strategy for Topic Identification

The first step in this project is to define a keyword set that can reliably identify YouTube videos related to the U.S. economy.

Because news channels publish a wide range of content, keyword selection is an important filtering step that helps distinguish economy-related videos from unrelated uploads. To improve coverage, the keyword list includes both broad economic terms and issue-specific phrases that are likely to appear in video titles. The keyword set may need refinement if the initial search returns too few relevant videos or captures too much off-topic content.

In this section, I document the selected keywords and explain why each one is useful for identifying videos related to the project topic.

My keywords are:
- **U.S. economy / U.S. economic:** This phrase directly captures videos explicitly framed around the U.S. economy.
- **Economy / Economic:** These are the most common broad terms used in news coverage of economic issues.
- **Inflation:** Inflation is one of the most visible and frequently discussed economic conditions in recent news.
- **Recession:** Recession is another major economic theme that often appears in coverage of economic outlook and policy.
- **Price / Prices:** Price-related language reflects economic pressure in a concrete, everyday form.
- **Job market:** The job market is one of the clearest public indicators of economic conditions.

### YouTube Data Collection Workflow

After defining the keyword strategy, the next step is to collect topic-relevant videos and comment-level data from YouTube.

This workflow includes:
- retrieving uploaded videos from selected YouTube channels
- filtering videos based on economy-related keywords in the title
- extracting metadata for relevant videos
- collecting comment-level engagement data
- organizing the results into a structured dataset for downstream analysis

The final output of this notebook is a comment-level data frame in which each row represents one YouTube comment associated with a topic-relevant video.

In [1]:
# initialize YouTube API
API_KEY = ""

!pip install --upgrade google-api-python-client --quiet

import json
import googleapiclient
import googleapiclient.discovery
import googleapiclient.errors
import re
import csv

youtube = googleapiclient.discovery.build("youtube", "v3", developerKey = API_KEY)

### Video Retrieval and Topic Filtering

In this section, I retrieve uploaded videos from the selected YouTube channels, starting from the most recent, and identify videos whose titles match the economy-related keyword set defined above.

The purpose of this step is to build a topic-focused subset of videos rather than working with all channel uploads. Matching is handled in a case-insensitive way, and simple term variations are treated as equivalent when appropriate so that relevant videos are not missed due to capitalization or minor wording differences.

For each matched video, I save key metadata that will be used later for analysis, including identifiers, titles, timestamps, and engagement-related attributes.

In [2]:
# list channels
# CNN has a legacy username, while Fox News doesn't so we need to search by channel name
def get_channels(channel_name):
    if channel_name == "CNN":
        return youtube.channels().list(
            part = "id", 
            forUsername = "cnn"  # legacy username
        ).execute()["items"][0]["id"]  # get CNN channel ID
    else:
        return youtube.search().list(
            part = "snippet",
            q = "Fox News",           
            type = "channel",      # return channels
            maxResults = 1         # output the top one (most relevant)
        ).execute()["items"][0]["snippet"]["channelId"]  # get Fox News channel ID

In [3]:
# collect the most recent 50 topic-related videos per channel
def collect_recent_50_videos(channel):
    channels = get_channels(channel)

    # get uploads of playlists on channel (by default from the most recent)
    uploads = youtube.channels().list(
        part = "contentDetails", 
        id = channels
    ).execute()["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

    video_list = []  # an empty list to store videos 
    page = None  # for pagination

    while len(video_list) < 50:
        playlists = youtube.playlistItems().list(
            part = "contentDetails",
            playlistId = uploads,
            maxResults = 50,
            pageToken = page
        ).execute()
        
        video_id = []
        for video in playlists["items"]:
            video_id.append(video["contentDetails"]["videoId"])

        # get videos
        videos = youtube.videos().list(
            part = "snippet,statistics",  #statistics to get number of views
            id = ",".join(video_id)  # pass multiple video IDs
        ).execute()
        # get titles of videos for key words match
        for video in videos["items"]:
            video_title = video["snippet"]["title"]

            if keywords.search(video_title):  # successful match         
                video_list.append({
                    "channel_name": channel,
                    "video_id": video["id"],
                    "video_title": video_title,
                    "video_creation_time": video["snippet"]["publishedAt"],
                    "video_number_of_views": video["statistics"].get("viewCount")
                })

                if len(video_list) >= 50:  # 50 done
                    break

        page = playlists.get("nextPageToken")  # otherwise get into the next page
        if not page:
            break
            
    return video_list

In [4]:
# keyword matching using regular expression 
# not case-sensitive and singular and plural forms should be treated as matches
keywords = re.compile(
    r"\bU\.?S\.?\s+econom(y|ic)\b|"      
    r"\beconom(y|ic)\b|"               
    r"\binflation\b|"                  
    r"\brecession\b|"                   
    r"\bprices?\b|"                     
    r"\bjob\s+market\b",               
    re.IGNORECASE
)

In [5]:
# make sure having identified at least 50 videos per channel
cnn_videos = collect_recent_50_videos("CNN")
print("CNN videos:", len(cnn_videos))

fox_news_videos = collect_recent_50_videos("Fox News")
print("Fox News videos:", len(fox_news_videos))

CNN videos: 50
Fox News videos: 50


In [6]:
# make sure having identified topic_related videos per channel
for video in cnn_videos[:5]:
    print(video["video_title"])

for video in fox_news_videos[:5]:
    print(video["video_title"])

Gold surges to record $5000 price
Prices may start to rise in 2026 due to Trump tariffs
Trump promotes economy amid rising cost-of-living concerns
Michigan voters weigh in as Trump talks economy in Detroit
Demonstrations spread in Iran over inflation crisis
Trump proposes state fuel tax cap to slash California gas prices
Trump speaks at World Economic Forum CEO dinner in Davos
We had a ‘sick economy’ after Biden, economist says
Trump delivers remarks at the Detroit Economic Club
Trump predicts Cuba 'going down' as experts discuss sanctions, economic impact


### Comment Extraction

Once the relevant videos have been identified, I collect comment-level data for each video using the YouTube API.

For consistency across videos, the extraction targets the most relevant comments returned by the API. If a video has fewer available comments than expected, I retain all comments that are available and note this as part of the data collection context. This step expands the project from video-level content collection into comment-level discourse analysis, which is the main focus of the downstream analysis notebooks.

In [7]:
# fetch the 30 most relevant (as sorted by the API) comments if it's for one video
def get_30_comments(video_id):
    
    comments = []  # an empty list to store comments
    
    try:
        comment_threads = youtube.commentThreads().list(
            part = "snippet",
            videoId = video_id,
            maxResults = 30,
            order = "relevance",  # most relevant
            textFormat = "plainText"  # get cleaner output
        ).execute()

        for comment_thread in comment_threads.get("items", []):
            comment = comment_thread["snippet"]["topLevelComment"]
            
            comments.append({
                "comment_id": comment["id"],                        
                "comment_title": comment["snippet"].get("textDisplay", ""),
                "comment_creation_time": comment["snippet"].get("publishedAt", ""),
                "comment_number_of_likes": comment["snippet"].get("likeCount", 0)  # if none, return 0
            })
            
    except googleapiclient.errors.HttpError:
        return []  # just in case some video comments are not accessible that it won't impact the whole loop

    return comments

In [8]:
# collect the most recent 50 topic-related videos per channel as well as their 30 most revelant comments
def collect_recent_50_videos(channel):
    channels = get_channels(channel)

    # get uploads of playlists on channel
    uploads = youtube.channels().list(
        part = "contentDetails", 
        id = channels
    ).execute()["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

    video_list = []  # an empty list to store videos 
    page = None  # for pagination

    while len(video_list) < 50:
        playlists = youtube.playlistItems().list(
            part = "contentDetails",
            playlistId = uploads,
            maxResults = 50,
            pageToken = page
        ).execute()
        
        video_id = []
        for video in playlists["items"]:
            video_id.append(video["contentDetails"]["videoId"])

        # get videos
        videos = youtube.videos().list(
            part = "snippet,statistics",  #statistics to get number of views
            id = ",".join(video_id)  # pass multiple video IDs
        ).execute()
        # get titles of videos for key words match
        for video in videos["items"]:
            video_title = video["snippet"]["title"]

            if keywords.search(video_title):  # successful match
                
                comments = get_30_comments(video["id"])  # get 30 comments
                
                video_list.append({
                    "channel_name": channel,
                    "video_id": video["id"],
                    "video_title": video_title,
                    "video_creation_time": video["snippet"]["publishedAt"],
                    "video_number_of_views": video["statistics"].get("viewCount"),
                    "comments": comments, 
                    "number_of_comments": len(comments)  # check if having 30 comments
                })

                if len(video_list) >= 50:  # 50 done
                    break

        page = playlists.get("nextPageToken")  # otherwise get into the next page
        if not page:
            break
            
    return video_list

In [9]:
cnn_videos = collect_recent_50_videos("CNN")
fox_news_videos = collect_recent_50_videos("Fox News")

In [10]:
# take a look at the comments number under each video
for i, v in enumerate(cnn_videos):
    print(i, v["video_title"], v["number_of_comments"])

0 Gold surges to record $5000 price 30
1 Prices may start to rise in 2026 due to Trump tariffs 30
2 Trump promotes economy amid rising cost-of-living concerns 30
3 Michigan voters weigh in as Trump talks economy in Detroit 30
4 Demonstrations spread in Iran over inflation crisis 30
5 We’re in a ‘windchill’ economy, where things feel worse than they are 30
6 Inflation cooled in November to 2.7% 30
7 The economy & inflation: Harry Enten breaks down public opinion 30
8 Trump blames Biden, Dems, Fed chair and immigrants for economic woes 30
9 The next economic shift 30
10 US proposes ‘free economic zone’ in parts of Donbas after Ukrainian pullback 30
11 How the White House is using misleading comparisons to make inflation sound better 30
12 Collins presses Leavitt about mixed messaging on the economy 30
13 Trump says he aced the economy. Hear what people at the grocery store think 30
14 Trump grades his economy an ‘A-plus-plus-plus-plus-plus' 30
15 ‘It’s rough out there’: Enten runs number

In [11]:
# take a look at the comments number under each video
for i, v in enumerate(fox_news_videos):
    print(i, v["video_title"], v["number_of_comments"])

0 Trump proposes state fuel tax cap to slash California gas prices 30
1 Trump speaks at World Economic Forum CEO dinner in Davos 30
2 We had a ‘sick economy’ after Biden, economist says 30
3 Trump delivers remarks at the Detroit Economic Club 30
4 Trump predicts Cuba 'going down' as experts discuss sanctions, economic impact 30
5 The economy is turning a corner, here’s why 2026 looks strong 30
6 Iran's economy has been mostly ruined by the ayatollah and his 'henchmen': Mike Pompeo 30
7 Americans are about to feel Trump’s economic impact: Brian Kilmeade 30
8 NEW DATA JUST IN: Here are the US states with HIGHER inflation 30
9 Democrats MELT DOWN as economy grows under Trump 30
10 '2026 FEAST': Here's where the economy stands heading into the new year... 30
11 ‘STUNNING!’: Hosts react to ‘phenomenal’ report on Trump’s economy 30
12 BREAKING: White House celebrates 'BLOCKBUSTER' economic report 30
13 This is a 'good thing' for the US economy: Kevin O'Leary 30
14 President Donald Trump deli

A small number of videos have fewer than 30 available comments. In most cases, however, the comment volume is sufficient to support a relatively consistent comment-level dataset across videos.

In [12]:
# take a look at the comments example
for comment in cnn_videos[0]["comments"][:5]:
    print(comment)

{'comment_id': 'Ugwe-foTG3moc3xxtrd4AaABAg', 'comment_title': 'America is toast', 'comment_creation_time': '2026-01-26T15:34:36Z', 'comment_number_of_likes': 98}
{'comment_id': 'Ugyou95rzVIIpfOBBpR4AaABAg', 'comment_title': "He's going for the YUGest bankruptcy of his life - the USA.", 'comment_creation_time': '2026-01-26T15:41:47Z', 'comment_number_of_likes': 35}
{'comment_id': 'Ugy1uAYCCiQLEkcgz4F4AaABAg', 'comment_title': 'Gold only surge when economic not stable', 'comment_creation_time': '2026-01-26T15:35:51Z', 'comment_number_of_likes': 44}
{'comment_id': 'UgzjmzQUiJf5Dyxz73Z4AaABAg', 'comment_title': 'Because the world doesn’t trust the US dollar anymore because of tRump. MAGA might not like it or admit it, but it’s the truth. tRump made the world economy a heck of allot less stable. And many economists predicted this. Kamala warned us all this would happen, but you know, she’s a woman so folks would rather believe the blatant lies of a man than a woman who tells the truth.', 'c

In [13]:
# take a look at the comments example
for comment in fox_news_videos[0]["comments"][:5]:
    print(comment)

{'comment_id': 'UgwbDjuEh5KECQWxLil4AaABAg', 'comment_title': 'Our roads are garbage they DO NOT FIX OUR ROADS! They ARE FILLED WITH POT HOLES!', 'comment_creation_time': '2026-01-28T16:28:22Z', 'comment_number_of_likes': 121}
{'comment_id': 'UgzCnGXqDdIji9JVFBh4AaABAg', 'comment_title': 'No, the state of California politicians want to line their pockets with all these taxes they impose on gasoline.', 'comment_creation_time': '2026-01-28T18:23:48Z', 'comment_number_of_likes': 85}
{'comment_id': 'UgxC8ucWlhgLW4xOgbZ4AaABAg', 'comment_title': 'The fact that the state of CA would sue the feds for trying to make things affordable for CA residents just tells you what democrats are really all about.', 'comment_creation_time': '2026-01-28T23:58:12Z', 'comment_number_of_likes': 52}
{'comment_id': 'UgwJzXdub3KLbsdzIr14AaABAg', 'comment_title': "Thank you Trump for  helping Californians, we don't seem to be getting any from our state legislature", 'comment_creation_time': '2026-01-28T17:14:41Z',

### Building an Analysis-Ready Data Frame

After collecting both video metadata and comment-level information, I organize the data into a Pandas data frame where each row represents one comment.

This structure makes the dataset suitable for downstream analysis because it preserves both the comment itself and the context of the video it belongs to. In addition to the required fields such as channel name, video identifiers, titles, timestamps, view counts, comment identifiers, and likes, I also retain any additional attributes that may be helpful for later exploratory analysis or text processing.

At the end of this section, I verify the size of the dataset and preview the first few rows to confirm that the structure is correct.

In [14]:
import pandas as pd

row_comment = []

for video in cnn_videos + fox_news_videos:
    for comment in video["comments"]:
        row_comment.append({
            "channel_name": video["channel_name"],
            "video_id": video["video_id"],
            "video_title": video["video_title"],
            "video_creation_time": video["video_creation_time"],
            "video_number_of_views": video["video_number_of_views"],
            "comment_id": comment["comment_id"],
            "comment_title": comment["comment_title"],
            "comment_creation_time": comment["comment_creation_time"],
            "comment_number_of_likes": comment["comment_number_of_likes"]
        })

yt_comments = pd.DataFrame(row_comment)

To ensure the dataset is large enough for downstream analysis, I verify the number of collected comment rows for each channel.

In [15]:
yt_comments.groupby('channel_name').size()

channel_name
CNN         1478
Fox News    1482
dtype: int64

After constructing the comment-level data frame, I inspect the first few rows to confirm that the structure and key fields were assembled correctly.

In [16]:
yt_comments.head()

,channel_name,video_id,video_title,video_creation_time,video_number_of_views,comment_id,comment_title,comment_creation_time,comment_number_of_likes
0,CNN,L5tvym36mO0,Gold surges to record $5000 price,2026-01-26T15:32:49Z,51869,Ugwe-foTG3moc3xxtrd4AaABAg,America is toast,2026-01-26T15:34:36Z,98
1,CNN,L5tvym36mO0,Gold surges to record $5000 price,2026-01-26T15:32:49Z,51869,Ugyou95rzVIIpfOBBpR4AaABAg,He's going for the YUGest bankruptcy of his li...,2026-01-26T15:41:47Z,35
2,CNN,L5tvym36mO0,Gold surges to record $5000 price,2026-01-26T15:32:49Z,51869,Ugy1uAYCCiQLEkcgz4F4AaABAg,Gold only surge when economic not stable,2026-01-26T15:35:51Z,44
3,CNN,L5tvym36mO0,Gold surges to record $5000 price,2026-01-26T15:32:49Z,51869,UgzjmzQUiJf5Dyxz73Z4AaABAg,Because the world doesn’t trust the US dollar ...,2026-01-26T15:44:13Z,91
4,CNN,L5tvym36mO0,Gold surges to record $5000 price,2026-01-26T15:32:49Z,51869,UgzZIZEgQGeu38XsEB14AaABAg,We were told this day would come! Invest in go...,2026-01-26T16:09:36Z,10


### Exporting the Dataset

The final step in this notebook is to export the cleaned and structured comment-level data frame to a CSV file.

Saving the dataset in CSV format makes it easy to reuse in the next stages of the project, including data cleaning, exploratory analysis, and text-based modeling. This exported file serves as the foundation for the later notebooks in the workflow.

In [17]:
yt_comments.to_csv("yt_comments.csv", index = False)  # not writing row index